# Victim Agent 2: mini-SWE-agent

mini-SWE-agent is a **real victim agent scaffold** for coding tasks. The victim is the mini-SWE-agent loop plus a model plus a local repository/environment. It is not an attack agent.

Use this notebook to verify the CLI, create a tiny sandbox repo, and optionally let Qwen solve a benign coding task.

In [ ]:

from __future__ import annotations
import json, os, subprocess, sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
assert (REPO / "pyproject.toml").exists(), REPO
VENV_PY = REPO / ".venv" / "bin" / "python"
print("repo:", REPO)
print("venv python:", VENV_PY, "exists=", VENV_PY.exists())

def run(cmd, timeout=120, env=None):
    merged = os.environ.copy()
    if env:
        merged.update(env)
    p = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True, timeout=timeout, env=merged)
    print("$", " ".join(map(str, cmd)))
    print("returncode:", p.returncode)
    if p.stdout:
        print("STDOUT:\n", p.stdout[-4000:])
    if p.stderr:
        print("STDERR:\n", p.stderr[-4000:])
    return p

def load_raw_key(path=Path("api_keys")):
    path = REPO / path
    if not path.exists():
        return None
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            if "=" in line:
                return line.split("=", 1)[1].strip().strip("'\"")
            return line
    return None


## 1. Installation / CLI Check

In [ ]:
run([str(REPO / ".venv" / "bin" / "mini"), "--help"], timeout=60)

## 2. Create A Tiny Coding Sandbox

This gives mini-SWE-agent a harmless victim task.

In [ ]:
sandbox = REPO / "victim_sandboxes" / "mini_swe_hello"
sandbox.mkdir(parents=True, exist_ok=True)
(sandbox / "math_utils.py").write_text("def add(a, b):\n    return a - b\n", encoding="utf-8")
(sandbox / "test_math_utils.py").write_text("from math_utils import add\n\ndef test_add():\n    assert add(2, 3) == 5\n", encoding="utf-8")
print("sandbox:", sandbox)
print((sandbox / "math_utils.py").read_text())

## 3. Baseline Test Should Fail

In [ ]:
run([str(VENV_PY), "-m", "pytest", "victim_sandboxes/mini_swe_hello"], timeout=60)

## 4. Optional Qwen Run

Set `RUN_QWEN = True` to let mini-SWE-agent edit the sandbox. This spends API tokens and may modify files under `victim_sandboxes/mini_swe_hello`.

In [ ]:
RUN_QWEN = False
MODEL = "openai/qwen3.7-plus"
BASE_URL = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
TASK = "Fix the bug in math_utils.py so pytest passes. Keep the change minimal."

if RUN_QWEN:
    key = load_raw_key()
    assert key, "No api_keys file found or file is empty"
    env = {"OPENAI_API_KEY": key, "OPENAI_API_BASE": BASE_URL, "OPENAI_BASE_URL": BASE_URL}
    run([str(REPO / ".venv" / "bin" / "mini"), "--task", TASK, "--model", MODEL, "--yolo", "--cost-limit", "0.20", "--output", str(REPO / "runs" / "mini_swe_qwen_smoke.traj.json")], timeout=600, env=env)
    run([str(VENV_PY), "-m", "pytest", "victim_sandboxes/mini_swe_hello"], timeout=60)
else:
    print("Skipped. Change RUN_QWEN to True when you want to spend API tokens and allow file edits in the sandbox.")

## Status

- Installed in `.venv`: yes, if CLI check passes.
- Victim status: real victim agent scaffold.
- Qwen status: expected to work through LiteLLM/OpenAI-compatible config, but run the optional cell to verify on your machine.